# DATALUS Training Factory: Massive Scale & Multi-GPU

This notebook demonstrates the **DATALUS Training Factory**, designed for heavy compute environments (e.g., Kaggle 2x T4 or AWS G4 instances). 
Unlike the POC, we focus strictly on massive data ingestion, Out-Of-Memory (OOM) safe Polars processing, 
Distributed Multi-GPU training using `torch.nn.DataParallel`, and Edge deployment export via Post-Training Quantization (PTQ).

The objective is to demonstrate the framework's scalability when handling millions of records from the Brazilian Health Information System (SIH-SUS).

## 1. Environment Setup
Dynamically detecting the environment (Kaggle, Colab, Local) and installing required packages.

In [ ]:
import os
import sys
from pathlib import Path

def get_working_dir():
    if 'google.colab' in sys.modules:
        return '/content/datalus_factory'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return '/kaggle/working/datalus_factory'
    return './datalus_factory'

WORKING_DIR = get_working_dir()
os.makedirs(WORKING_DIR, exist_ok=True)
print(f'Working directory: {WORKING_DIR}')

# Install dependencies if in Colab/Kaggle
if 'google.colab' in sys.modules or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    !git clone https://github.com/emanuellcs/datalus.git || true
    %cd datalus
    !python -m pip install -e '.[dev]' pysus polars
    sys.path.append(os.getcwd())


## 2. Heavy Data Ingestion (SIH-SUS 2023 Full Year)
We simulate real-world scale by downloading an entire year of SIH-SUS data (São Paulo, 2023) using the `pysus` API. 
Saving the concatenated chunks to a single massive Parquet file allows the DATALUS Polars engine to utilize **lazy evaluation** (`pl.scan_parquet`). 

This architectural choice guarantees that we avoid RAM exhaustion even when processing datasets that exceed available memory, as data is streamed efficiently to the tensor encoding layer.

In [ ]:
from pysus.api import sih
import polars as pl

print('Downloading 12 months of SIH-SUS data for SP (2023)...')
# Scalable download and concatenation
dfs = []
for m in range(1, 13):
    print(f'Fetching month {m}...')
    df_m = sih(state='SP', year=2023, month=[m])
    dfs.append(pl.from_pandas(df_m))

df_massive = pl.concat(dfs, how='vertical_relaxed')
df_massive = df_massive.with_columns(
    pl.col('MORTE').cast(pl.Int64).fill_null(0)
)

cols_to_keep = [c for c in df_massive.columns if not c.startswith(('N_AIH', 'IDENT'))]
df_massive = df_massive.select(cols_to_keep)

raw_path = f'{WORKING_DIR}/massive_raw_sih.parquet'
df_massive.write_parquet(raw_path)
print(f'Massive dataset saved to {raw_path} | Shape: {df_massive.shape}')

## 3. High-Throughput Ingestion
The `ingest` command maps the raw massive Parquet to the tensor-ready format, handling categorical frequencies and numeric quantile transforms.

In [ ]:
!datalus ingest {WORKING_DIR}/massive_raw_sih.parquet {WORKING_DIR}/processed.parquet --schema-path {WORKING_DIR}/schema_config.json --target-column MORTE

## 4. Distributed Multi-GPU Training Factory
This is the core compute phase. We pass `--gpu "0,1"` to distribute the load across multiple GPUs. 
DATALUS automatically masks environment variables and wraps the residual MLP denoiser in `torch.nn.DataParallel` for scalable throughput.

**Key Components:**
- **Cosine Schedule:** Sophisticated diffusion variance control.
- **EMA Tracking:** Exponential Moving Average of weights to ensure structural stability.
- **Mixed Precision (AMP):** Faster training with lower VRAM footprint.
- **Deterministic Checkpointing:** Resumable training states across the distributed cluster.

In [ ]:
!datalus train {WORKING_DIR}/schema_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/artifacts --epochs 50 --batch-size 2048 --checkpoint-every-steps 1000 --gpu "0,1"

## 5. Artifact Decoupling: Edge Export
After training on high-end hardware, we decouple the generative model. 
Using Post-Training Quantization (**INT8 PTQ**), we compress the massive float32 diffusion model.

This allows the model, despite being trained on 2x T4 GPUs, to run smoothly on local CPUs or directly in the browser via ONNX Runtime Web.

In [ ]:
!datalus export-onnx {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/artifacts --quantize